**1. Upload Data**

In [ ]:
# Upload Parkinson disease.csv from UCI Machine Learning Repository

# AI use: Claude was used during this project to help structure and debug code.
# Verified the correctness of solutions using concepts learned in class and by
# checking that outputs and results aligned with our own understanding of the material.
from google.colab import files

uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

**2. Preprocessing of the Data**

In [ ]:

# 2. LOAD AND PREPROCESS DATA

import pandas as pd
import numpy as np

# Load uploaded CSV file
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)

# Extract subject ID before dropping the name column
# This is important because the dataset has multiple recordings per subject.
# Example: phon_R01_S01_1 → subject ID = S01

#we are keeping the ID's since each patient has 6 recordings and this way there isn't leakage
#each patient recording is a row so there's 195 rows but there's only 31 people
#without remembering IDs then the same patient could be in both the training and testing folds when doing CV later
#this way we can keep all of the same person's trials in either the training or the left out fold.
groups = df_raw['name'].apply(lambda x: x.split('_')[2])

# Target variable
# status = 1 means Parkinson's
# status = 0 means Healthy control
y = df_raw['status']

# All 22 voice features for logistic regression
X_all = df_raw.drop(columns=['name', 'status'])

# Feature set used for the neural network model (chose 10 best for neural network instead of all)
nn_features = [
    'MDVP:Shimmer(dB)',
    'MDVP:Jitter(%)',
    'HNR',
    'NHR',
    'RPDE',
    'DFA',
    'spread1',
    'spread2',
    'D2',
    'PPE'
]

X_nn = df_raw[nn_features]

# Display basic dataset information
print("Dataset loaded successfully.")
print("Dataset shape:", df_raw.shape)
print("Number of voice recordings:", len(df_raw))
print("Number of unique subjects:", groups.nunique())
print("\nClass distribution:")
print(y.value_counts())

print("\nAll voice features:")
print(list(X_all.columns))

print("\nNeural network selected features:")
print(nn_features)

**3. PCA/Correlations**

In [ ]:
# ================================
# PCA EXPLAINED VARIANCE
# ================================

# In figuring out number of optimal PCs, we figured to not feed the neural network all 22 features as to not overfit
# Decided to only give 10 features to neural network for simplicity. Didn't give pca components to nn for simpler use in clinical setting.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Standardize features by scaling each of the features to mean of 0 and std 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# PCA
pca = PCA()
pca.fit(X_scaled) #finds best PCs

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

plt.figure(figsize=(8,5))

# This graph will help us figure out the optimal number of PCs to use
# Shows at which PC that the variance threshold is hit as to not have repeated information
plt.plot(
    range(1, len(cumulative_var)+1),
    cumulative_var,
    marker='o'
)

plt.axhline(
    y=0.90,
    linestyle='--',
    label='90% Variance'
)

plt.axhline(
    y=0.95,
    linestyle='--',
    label='95% Variance'
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Cumulative Explained Variance")
plt.legend()

plt.show()

print("Principal Components needed for 90% variance:",
      np.argmax(cumulative_var >= 0.90) + 1)

print("Principal Components needed for 95% variance:",
      np.argmax(cumulative_var >= 0.95) + 1)

In [ ]:
# ================================
# PCA SCATTER PLOT
# ================================
# Visualization for us to ensure the two groups actually differ
# Since the clusters of healthy vs. parkinson's patients were separated a bit on PC1 it gave us confidence
# that there was some features that would allow us to distinguish status.
# If there wasn't any difference on PC1 vs PC2 plot, that would show us there isn't much pattern in status variance in the factors. Was not the case here.

pca_2d = PCA(n_components=2)

X_pca = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[y==0,0],
    X_pca[y==0,1],
    label='Healthy',
    alpha=0.7
)

plt.scatter(
    X_pca[y==1,0],
    X_pca[y==1,1],
    label="Parkinson's",
    alpha=0.7
)

plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection of Voice Features")
plt.legend()

plt.show()

In [ ]:
# ================================
# FEATURE CORRELATION MATRIX
# ================================
# Helped us visualize how much each feature was correlated to other features in order to figure out duplicate features that provide the same information.
# This helped us see how all the different jitter and shimmer variants would rise and fall together.
# It helped us figure out redundant factors and narrow down to the 10 most relevent factors.
# There was no need to include factors that gave the same information as to not overfit.
# For example, shimmer is one of the most telling factors but it has several variants.
# Rather than using the "top 10 most correlated" factors, which would all just be variants of shimmer that give the same information,
# we were able to figure out which ones 10 together would provide the most unique information.

import seaborn as sns

plt.figure(figsize=(12,10))

corr = X_all.corr()

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0
)

plt.title("Voice Feature Correlation Matrix")

plt.show()

**4. Cross-Validation**

In [ ]:
# ================================
# 4. GROUPED 5-FOLD CROSS-VALIDATION
# ================================
# We used 5-fold CV to test our model split and accuracy. This code is just the cv 5-fold design that's later utilized in both model types.
# The 5-fold CV breaks the data in 5 different folds, training on 4 of them then testing with the fifth one. Then runs 5 times so that each one is a test set and averages the 5 accuracy scores.
# We used stratified group in order to not leak data by keeping all 6 voice recordings of each person together.
# Originally we didn't do this and our model predicted way higher accuracy but then we realized it was actually leaking data by treating each of the voice trials as independent.
# The stratified part keeps the class balance even which is important since our number of Parkinson's vs. healhty patients is about a 3:1 ratio.

from sklearn.model_selection import StratifiedGroupKFold

# The dataset contains multiple recordings per subject.
# To prevent data leakage, recordings from the same
# individual must remain in the same fold.

gkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

print("Cross-validation method:")
print("StratifiedGroupKFold")

print("\nNumber of folds:", gkf.n_splits)

print("\nPurpose:")
print("- Prevent subject-level data leakage")
print("- Maintain Parkinson's/Healthy class balance")
print("- Provide a more reliable estimate of model performance")

**5. Neural Network**

In [ ]:
# ================================
# 5. NEURAL NETWORK MODEL
# Grouped 5-Fold CV to prevent subject-level data leakage
# ================================

# Building of our model, with 6 different layers.
# First input of only the 10 features we chose.
# The Dense(64, relu) uses 64 neurons to look for the patterns and ReLu to make the realtionships non-linear. This is important since linear data just converges on itself.
# Dropout switches randomly switchs off some of the neurons to reduce overfitting
# Thesecond dense and dropout layers do the same as the first just in another layer
# Then the final layer is the singular output using sigmoid to output a number between the 0-1 range for probability

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, roc_auc_score, accuracy_score, f1_score

tf.random.set_seed(50)
np.random.seed(50)

# Use the selected neural network features from Part 2
X = X_nn.values
y_arr = y.values

# Build neural network
def build_nn(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

nn_true = []
nn_probs = []

nn_fold_aucs = []
nn_fold_accs = []
nn_fold_f1s = []

# This runs the CV that was described and written earlier. Everything repeats 5 times, and each time the CV runs it produces the 5 folds.
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_arr, groups), 1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_arr[train_idx], y_arr[test_idx]

    # Scale inside each fold to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = build_nn(input_dim=X_train_scaled.shape[1])

    model.fit(
        X_train_scaled,
        y_train,
        epochs=100,
        batch_size=16,
        validation_data=(X_test_scaled, y_test),
        verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True
            )
        ]
    )

    y_prob = model.predict(X_test_scaled, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    # Guard against folds with only one class
    # Makes sure that a single fold doesn't contain just one status so all Parkinson's patients or all healthy patients
    # This way AUC is always computed and doesn't crash
    if len(np.unique(y_test)) < 2:
        print(f"Fold {fold} — skipping AUC (only one class in test set)")
        fold_auc = np.nan
    else:
        fold_auc = roc_auc_score(y_test, y_prob)

    fold_acc = accuracy_score(y_test, y_pred)
    fold_f1 = f1_score(y_test, y_pred)

    nn_fold_aucs.append(fold_auc)
    nn_fold_accs.append(fold_acc)
    nn_fold_f1s.append(fold_f1)

    nn_true.extend(y_test)
    nn_probs.extend(y_prob)

    print(f"Fold {fold}")
    print(f"AUC: {fold_auc:.3f}" if not np.isnan(fold_auc) else "AUC: n/a (single class)")
    print(f"Accuracy: {fold_acc:.3f}")
    print(f"F1 Score: {fold_f1:.3f}")
    print("Test subjects:", groups.iloc[test_idx].unique())
    print()

# Overall grouped CV ROC/AUC
nn_true = np.array(nn_true)
nn_probs = np.array(nn_probs)

# This combines all the predictions from the 5 folds to build the ROC curve and find AUC for each of the prediction fold
# AUC will tell us the accuracy of the model on scale of 0 to 1
nn_fpr, nn_tpr, nn_thresholds = roc_curve(nn_true, nn_probs)
nn_auc = auc(nn_fpr, nn_tpr)

# Filter out nan before computing mean
valid_aucs = [a for a in nn_fold_aucs if not np.isnan(a)]

print("===== Neural Network Grouped CV Summary =====")
print(f"Mean Fold AUC (valid folds): {np.mean(valid_aucs):.3f} ± {np.std(valid_aucs):.3f}")
print(f"Mean Accuracy: {np.mean(nn_fold_accs):.3f} ± {np.std(nn_fold_accs):.3f}")
print(f"Mean F1 Score: {np.mean(nn_fold_f1s):.3f} ± {np.std(nn_fold_f1s):.3f}")
print(f"Overall Grouped Cross-Validated AUC: {nn_auc:.3f}")

# The ROC curve shows the model probability across the possible thresholds and the dashed line is what would occur with random chance guessing
# The curve itself is demonstrating probability by evaluating the model at each threshold so then we can AUC to compare between models
plt.figure(figsize=(7, 7))

plt.plot(
    nn_fpr,
    nn_tpr,
    linewidth=2,
    label=f'Grouped 5-Fold CV AUC = {nn_auc:.3f}'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Neural Network ROC Curve with Grouped 5-Fold Cross-Validation")
plt.legend()
plt.show()

In [ ]:
# ================================
# UNIT TEST: Single-Class Fold Guard
# Verifies the safeguard that handles a CV fold whose test set
# contains only one class (all Parkinson's or all healthy).
# ================================

# The point of this unit test is to check that the model doesn't crash if the random split of the folds creates a fold that is purely made up of patients of a single status
# Originally, this was an issue that would crash our code. to fix this, we had to add a safeguard in both the neural network and the logistic regression.
# The safeguard protects from the single status folds because before it would crash. It would cause AUC to be undefined since there was no ability to separate the two classes since there was only one class
# Before, when this case was sent, it messed up finding the ROC and AUC of that specific fold run and in doing so also mess up the average AUC

import unittest
import numpy as np
from sklearn.metrics import roc_auc_score

# The "repaired" logic, isolated so it can be tested on its own.
# This mirrors the guard used inside the neural network CV loop.
def safe_auc(y_true, y_prob):
    """Return ROC AUC for a fold, or np.nan if the test set has only one class."""
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


class TestSingleClassFoldGuard(unittest.TestCase):

    def test_all_parkinsons_fold_returns_nan(self):
        # Fold where every test recording is Parkinson's (status = 1)
        self.assertTrue(np.isnan(safe_auc([1, 1, 1, 1], [0.9, 0.8, 0.95, 0.7])))

    def test_all_healthy_fold_returns_nan(self):
        # Fold where every test recording is healthy (status = 0)
        self.assertTrue(np.isnan(safe_auc([0, 0, 0, 0], [0.1, 0.2, 0.05, 0.3])))

    def test_normal_fold_returns_valid_auc(self):
        # Control case: a normal fold with both classes still returns a real AUC.
        # Proves the guard only fires on the broken case, not on everything.
        result = safe_auc([0, 0, 1, 1], [0.1, 0.4, 0.6, 0.9])
        self.assertFalse(np.isnan(result))
        self.assertTrue(0.0 <= result <= 1.0)


# In a notebook, run the suite like this (argv/exit args prevent it from
# trying to parse the notebook's command-line args and killing the kernel):
unittest.main(argv=[""], exit=False, verbosity=2)

# ================================
# UNIT TEST: Safe AUC Edge Cases
# Verifies that safe_auc handles boundary conditions
# where one class is nearly absent from a CV fold.
# ================================

# The point of this unit test is to check that safe_auc still returns a valid number
# when only one sample of a class is present in the fold, not zero (which triggers nan),
# but just barely enough to compute AUC.

# Originally we only tested the all-one-class case, but in practice CV folds can also
# produce heavily skewed folds with just one positive or one negative sample.
# These edge cases should still return a real AUC since both classes are technically present.
# We also test perfect separation to confirm safe_auc returns 1.0 when the model is ideal.

class TestSafeAucEdgeCases(unittest.TestCase):

    def test_single_positive_returns_valid_auc(self):
        # Only one Parkinson's sample among many healthy — both classes present so should work
        result = safe_auc([0, 0, 0, 1], [0.1, 0.2, 0.3, 0.9])
        self.assertFalse(np.isnan(result))
        self.assertTrue(0.0 <= result <= 1.0)

    def test_single_negative_returns_valid_auc(self):
        # Only one healthy sample among many Parkinson's — mirror of above
        result = safe_auc([1, 1, 1, 0], [0.9, 0.8, 0.7, 0.1])
        self.assertFalse(np.isnan(result))
        self.assertTrue(0.0 <= result <= 1.0)

    def test_perfect_separation_returns_one(self):
        # If the model perfectly separates the two classes, AUC should be exactly 1.0
        result = safe_auc([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9])
        self.assertAlmostEqual(result, 1.0)

unittest.main(argv=[""], exit=False, verbosity=2)

In [ ]:
# ================================
# INTEGRATION TEST: Neural Network Pipeline
# Tests that preprocessing, grouped split, scaling, model training,
# prediction, and AUC safeguard work together end-to-end.
# ================================

# The unit test above only checks the AUC guard by itself.
# This integration test checks whether the major pieces of the model pipeline work together:
#1. selected input features are pulled correctly
#2. grouped CV split keeps each subject only in train OR test
#3. scaler fits only on train data and transforms test data
#4. neural network can train and output probabilities
#5. predicted probabilities are valid values between 0 and 1
#6. safe_auc handles the fold result without crashing

import unittest
import numpy as np
from sklearn.preprocessing import StandardScaler

class TestNeuralNetworkPipelineIntegration(unittest.TestCase):

    def test_one_grouped_fold_runs_end_to_end(self):
        # Use the same selected features, labels, groups, CV splitter,
        # build_nn() function, and safe_auc() helper used in the notebook.
        X_integration = X_nn.values
        y_integration = y.values

        # Take the first grouped fold as a lightweight integration test.
        train_idx, test_idx = next(gkf.split(X_integration, y_integration, groups))

        # Confirm there is no leakage between train and test.
        train_subjects = set(groups.iloc[train_idx])
        test_subjects = set(groups.iloc[test_idx])
        self.assertTrue(train_subjects.isdisjoint(test_subjects))

        # Scale inside the fold.
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_integration[train_idx])
        X_test_scaled = scaler.transform(X_integration[test_idx])

        self.assertEqual(X_train_scaled.shape[1], len(nn_features))
        self.assertEqual(X_test_scaled.shape[1], len(nn_features))

        # Train a small/short version of the same neural network pipeline.
        # epochs is kept low so the integration test runs quickly.
        test_model = build_nn(input_dim=X_train_scaled.shape[1])
        test_model.fit(
            X_train_scaled,
            y_integration[train_idx],
            epochs=2,
            batch_size=16,
            verbose=0
        )

        # Predict probabilities on the held-out grouped test fold.
        y_prob_test = test_model.predict(X_test_scaled, verbose=0).ravel()

        # Confirm prediction shape and probability range are valid.
        self.assertEqual(len(y_prob_test), len(test_idx))
        self.assertTrue(np.all(y_prob_test >= 0))
        self.assertTrue(np.all(y_prob_test <= 1))

        # Confirm AUC helper returns either a valid AUC or np.nan for single-class folds.
        fold_auc = safe_auc(y_integration[test_idx], y_prob_test)

        if len(np.unique(y_integration[test_idx])) < 2:
            self.assertTrue(np.isnan(fold_auc))
        else:
            self.assertTrue(0.0 <= fold_auc <= 1.0)


# Run only the integration test class so the unit tests above do not rerun.
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNeuralNetworkPipelineIntegration)
unittest.TextTestRunner(verbosity=2).run(suite)


In [ ]:
# ================================
# Neural Network Probability Distribution
# ================================
# Histogram shows the confidence level that the model has for its predictions of every single recording.
# Shows how many recordings were at each confidence level.
# The recordings at 0.5 means the model is very unsure. Most of the recordings it knows with 95-100% certainty but a good amount ~35, it was unsure.
plt.figure(figsize=(8,5))

plt.hist(
    nn_probs,
    bins=20
)

plt.xlabel("Predicted Parkinson's Probability")
plt.ylabel("Number of Recordings")

plt.title(
    "Neural Network Predicted Probability Distribution"
)

plt.show()

**6. Logistic Regression**

In [ ]:
# ================================
# 6. LOGISTIC REGRESSION MODEL
# Grouped 5-Fold CV to prevent subject-level data leakage
# ================================
# Thisis our baseline model since we wanted to be able to compare the performance of our neural network model against another model type.
# Uses the same 5-fold CV testing process as the neural network model, using the same CV system described earlier
# Since they were evaluated the same, we can compare the two models with their ROC curves and AUC.
# Also contains the same guard that stops a fold in the CV from containing only patients of the same class

# Logistic regression model input is all 22 features rather than just the most important 10. This is because overfitting is less of a concern with the simpler model.
# Used basic logistic regression, no layers like neural network has


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, roc_auc_score, accuracy_score, f1_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Logistic Regression uses all 22 voice features. this is INPUT
# Model output is the model itself -> most importantly the coefficients of each feature is what the model is solving for in order to output prediction and confidence
X = X_all
y_arr = y

lr_true = []
lr_probs = []

lr_fold_aucs = []
lr_fold_accs = []
lr_fold_f1s = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_arr, groups), 1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y_arr.iloc[train_idx]
    y_test = y_arr.iloc[test_idx]

    # Scale inside each fold to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build logistic regression model
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=50
    )

    lr_model.fit(X_train_scaled, y_train)

    y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # Guard against folds with only one class
    if len(np.unique(y_test)) < 2:
        print(f"Fold {fold} — skipping AUC (only one class in test set)")
        fold_auc = np.nan
    else:
        fold_auc = roc_auc_score(y_test, y_prob)

    fold_acc = accuracy_score(y_test, y_pred)
    fold_f1 = f1_score(y_test, y_pred)

    lr_fold_aucs.append(fold_auc)
    lr_fold_accs.append(fold_acc)
    lr_fold_f1s.append(fold_f1)

    lr_true.extend(y_test)
    lr_probs.extend(y_prob)

    print(f"Fold {fold}")
    print(f"AUC: {fold_auc:.3f}" if not np.isnan(fold_auc) else "AUC: n/a (single class)")
    print(f"Accuracy: {fold_acc:.3f}")
    print(f"F1 Score: {fold_f1:.3f}")
    print("Test subjects:", groups.iloc[test_idx].unique())
    print()

# Overall grouped CV ROC/AUC
lr_true = np.array(lr_true)
lr_probs = np.array(lr_probs)

lr_fpr, lr_tpr, lr_thresholds = roc_curve(lr_true, lr_probs)
lr_auc = auc(lr_fpr, lr_tpr)

# Use only valid folds for mean reporting
valid_aucs = [a for a in lr_fold_aucs if not np.isnan(a)]

print("===== Logistic Regression Grouped CV Summary =====")
print(f"Mean Fold AUC (valid folds): {np.mean(valid_aucs):.3f} ± {np.std(valid_aucs):.3f}")
print(f"Mean Accuracy: {np.mean(lr_fold_accs):.3f} ± {np.std(lr_fold_accs):.3f}")
print(f"Mean F1 Score: {np.mean(lr_fold_f1s):.3f} ± {np.std(lr_fold_f1s):.3f}")
print(f"Overall Grouped Cross-Validated AUC: {lr_auc:.3f}")

# ROC curve
plt.figure(figsize=(7, 7))

plt.plot(
    lr_fpr,
    lr_tpr,
    linewidth=2,
    label=f'Grouped 5-Fold CV AUC = {lr_auc:.3f}'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression ROC Curve with Grouped 5-Fold Cross-Validation")
plt.legend()
plt.show()

In [ ]:
# ================================
# LOGISTIC REGRESSION FEATURE IMPORTANCE
# Final full-data model for interpretation only
# ================================
# Logistic regression model refits log regression on all of the data then has one coefficient per feature that shows amount of weight per variable
# Since each feature was scaled already with a 0 to 1, by comparing the coefficients, you can already figure out which is weighed more with direct comparison.

final_lr_scaler = StandardScaler()
X_all_scaled = final_lr_scaler.fit_transform(X_all)

final_lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

final_lr_model.fit(X_all_scaled, y)

coefficients = final_lr_model.coef_[0]

lr_importance = pd.DataFrame({
    'Feature': X_all.columns,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

lr_importance = lr_importance.sort_values(
    by='Abs_Coefficient',
    ascending=False
)

print(lr_importance)

plt.figure(figsize=(10, 6))

plt.barh(
    lr_importance['Feature'].head(10)[::-1],
    lr_importance['Abs_Coefficient'].head(10)[::-1]
)

plt.xlabel("Absolute Coefficient Value")
plt.ylabel("Feature")
plt.title("Top Logistic Regression Feature Contributions")

plt.show()

**7. Comparison of Neural Network and Logistic Regression**

In [ ]:
# ================================
# 7. MODEL COMPARISON
# Neural Network vs Logistic Regression
# ================================
# Overlays the two ROC curves on one plot and shows the two AUC values against each other


import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

plt.plot(
    nn_fpr,
    nn_tpr,
    linewidth=2,
    label=f'Neural Network (AUC = {nn_auc:.3f})'
)

plt.plot(
    lr_fpr,
    lr_tpr,
    linewidth=2,
    color='red',
    label=f'Logistic Regression (AUC = {lr_auc:.3f})'
)

plt.plot(
    [0,1],
    [0,1],
    linestyle='--',
    color='orange'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "Neural Network vs Logistic Regression\nGrouped 5-Fold Cross-Validation"
)

plt.legend()

plt.show()

print("===== Model Comparison =====")
print(f"Neural Network AUC: {nn_auc:.3f}")
print(f"Logistic Regression AUC: {lr_auc:.3f}")

if nn_auc > lr_auc:
    print("\nSelected Model: Neural Network")
    print("Reason: Higher cross-validated AUC indicates better discrimination between Parkinson's and healthy patients.")
else:
    print("\nSelected Model: Logistic Regression")

**8. Final Model Selector**

In [ ]:
# ================================
# 8. FINAL MODEL SELECTION
# ================================
# Compares the two models against each other and just tells us that neural network is better based off AUC


print("===== Final Model Selection =====")
print(f"Neural Network AUC: {nn_auc:.3f}")
print(f"Logistic Regression AUC: {lr_auc:.3f}")

if nn_auc > lr_auc:
    final_model_name = "Neural Network"
    final_model_auc = nn_auc
    print("\nSelected Final Model: Neural Network")
    print("Reason: The Neural Network achieved the higher grouped cross-validated AUC.")
    print("This indicates stronger discrimination between Parkinson's and healthy patients.")
else:
    final_model_name = "Logistic Regression"
    final_model_auc = lr_auc
    print("\nSelected Final Model: Logistic Regression")
    print("Reason: Logistic Regression achieved the higher grouped cross-validated AUC.")

print("\nFinal model stored as:", final_model_name)
print(f"Final model AUC: {final_model_auc:.3f}")

**9. Final Parkinson's Disease Predictor**

In [ ]:
# ================================
# 9. FINAL PARKINSON'S PREDICTOR
# Train final Neural Network on all available data
# ================================

# This is the final neural network but instead it's trained on all of the data instead of leaving out any of the folds.
# This is the best possible predictor since it is trained on all of it but we can't know a true final accuracy unless there was additional data to test it out on.

from sklearn.preprocessing import StandardScaler

# Use the same selected neural network features
X_final = X_nn.values
y_final = y.values

# Scale all training data
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final)

# Build final neural network using same architecture
final_nn_model = build_nn(input_dim=X_final_scaled.shape[1])

# Train final model on all available data
final_nn_model.fit(
    X_final_scaled,
    y_final,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Final Neural Network Parkinson's predictor trained successfully.")

In [ ]:
# ================================
# Prediction Function
# ================================

def predict_parkinsons(new_patient_features):
    """
    Predicts Parkinson's disease probability using the final neural network model.

    Parameters:
        new_patient_features (dict): Dictionary containing the required vocal features.

    Returns:
        probability (float): Predicted probability of Parkinson's disease.
        classification (str): Predicted class label.
    """

    input_df = pd.DataFrame([new_patient_features])

    # Keep features in the same order used during training
    input_df = input_df[nn_features]

    # Scale using the final scaler
    input_scaled = final_scaler.transform(input_df)

    # Predict probability
    probability = final_nn_model.predict(input_scaled, verbose=0)[0][0]

    # Classify using 0.5 threshold
    classification = "Parkinson's" if probability >= 0.5 else "Healthy"

    print(f"Predicted Parkinson's Probability: {probability * 100:.2f}%")
    print(f"Predicted Classification: {classification}")

    return probability, classification

In [ ]:
# ================================
# Example Patient Prediction
# ================================
# This is fake patient data that we made up

example_patient = {
    'MDVP:Shimmer(dB)': 0.426,
    'MDVP:Jitter(%)': 0.00784,
    'HNR': 21.033,
    'NHR': 0.02211,
    'RPDE': 0.414783,
    'DFA': 0.815285,
    'spread1': -4.813031,
    'spread2': 0.266482,
    'D2': 2.301442,
    'PPE': 0.284654
}

predict_parkinsons(example_patient)

In [ ]:
example_patient = {
    'MDVP:Shimmer(dB)': 0.155000,
    'MDVP:Jitter(%)': 0.003460,
    'HNR': 26.143000,
    'NHR': 0.004150,
    'RPDE': 0.361232,
    'DFA': 0.763242,
    'spread1': -6.016891,
    'spread2': 0.109256,
    'D2': 2.004719,
    'PPE': 0.174429
}

predict_parkinsons(example_patient)